In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys 
sys.path.append('../src')

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [62]:
import torch
import tqdm
import matplotlib.pyplot as plt
from rescue.mapanything_pipeline import run_mapanything, save_points_to_glb, save_mesh_to_glb, save_language_features
from rescue.feature_reduction import TorchIncrementalPCA
from rescue.lang_features import LSegLangFeatures

device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
predictions = run_mapanything('../generated/long_pattern.mp4', '../generated/long_pattern.json', fps = 2, num_views = 10)

Loading MapAnything model...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading pretrained dinov2_vitg14 from torch hub


Using cache found in /home/srohatgi/.cache/torch/hub/facebookresearch_dinov2_main


Loading weights from local directory
Loading images...
Running MapAnything model...
MapAnything model run complete...


In [10]:
save_points_to_glb(predictions, "../generated/reconstruction_points.glb")
save_mesh_to_glb(predictions, "../generated/reconstruction_mesh.glb")

Building GLB scene
Using Pointmap Branch
GLB Scene built
Building GLB scene
Using Pointmap Branch
Creating mesh for multi-frame data...
GLB Scene built


In [18]:
lang_feats = LSegLangFeatures(checkpoint_path="../generated/lseg_minimal_e200.ckpt",)


In [1]:
img_list = [pred['img_no_norm'] for pred in predictions]
ipca = TorchIncrementalPCA(n_components=64, device='cuda')
language_features = []

for img in tqdm.tqdm(img_list):
    img_perm = torch.permute(img, (0, 3, 1, 2))
    img_resized = torch.nn.functional.interpolate(
        img_perm, size=(480, 640), mode="bilinear", align_corners=False
    )
    img_feat = lang_feats.extract_dense_from_tensor(img_resized)
    img_feat = torch.nn.functional.interpolate(
        img_feat, size=(img.shape[1], img.shape[2]), mode="bilinear", align_corners=False
    )
    img_feat = torch.permute(img_feat.squeeze(0), (1, 2, 0))
    language_features.append(img_feat)

language_features = torch.stack(language_features, dim=0)  # (S, H, W, C)

NameError: name 'predictions' is not defined

In [84]:
save_language_features(predictions, language_features, "../generated/language_features.npz")